# CMIP6 decadal daily concat and subset with batched outputs

**Date:** 2026-08-21

This example demonstrates that daily **CMIP6 decadal** data can be concatenated by realization and then subset through the CDS Rook service. The request selects the autumn months (September–November) of 1962–1964 over Europe from ten EC-Earth3 hindcast realizations.

Large collections are processed in batches. Consequently, one workflow response can contain several NetCDF output files. The cells below show how to discover, download, and inspect every file instead of assuming that a request always returns one file.

## Configure Rooki

In [1]:
import os
from time import perf_counter

os.environ["ROOK_URL"] = "http://rook.dkrz.de/wps"

import xarray as xr

from rooki import operators as ops

## Select the daily decadal collection

Each dataset identifier represents one realization initialized in 1961. The workflow first concatenates them along the `realization` dimension and then applies the spatiotemporal subset. This is the relevant decadal workflow and exercises the server-side batching used to keep the operation within its memory limits.

Rook's `area` order is `west,south,east,north`; the bounding box below covers Europe.

In [2]:
dataset_ids = [
    "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r1i1p1f1.day.psl.gr.v20201215",
    "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r2i1p1f1.day.psl.gr.v20201215",
    "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r3i1p1f1.day.psl.gr.v20201215",
    "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r4i1p1f1.day.psl.gr.v20201216",
    "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r5i1p1f1.day.psl.gr.v20201216",
    "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r6i1p1f1.day.psl.gr.v20201216",
    "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r7i1p1f1.day.psl.gr.v20201216",
    "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r8i1p1f1.day.psl.gr.v20201216",
    "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r9i1p1f1.day.psl.gr.v20201216",
    "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r10i1p1f1.day.psl.gr.v20201216",
]

time_range = "1962/1964"
autumn_months = "month:09,10,11"
europe = "-10,30,35,70"

## Submit the concat + subset workflow

The result is a single workflow response whose Metalink document describes all batch output files. The elapsed time measures the complete server-side orchestration as observed by the client.

In [3]:
psl = ops.Input("psl", dataset_ids)
concatenated = ops.Concat(psl, dims="realization")
workflow = ops.Subset(
    concatenated,
    time=time_range,
    time_components=autumn_months,
    area=europe,
)

started_at = perf_counter()
response = workflow.orchestrate()
elapsed_seconds = perf_counter() - started_at

print(f"Orchestration time: {elapsed_seconds:.1f} seconds ({elapsed_seconds / 60:.1f} minutes)")
print(response.status)
assert response.ok, response.status

Orchestration time: 30.8 seconds (0.5 minutes)
ProcessSucceeded


## Show the batched output URLs

The workflow response contains one download URL for each batched NetCDF output file.

In [4]:
for url in response.download_urls():
    print(url)

http://rook7.cloud.dkrz.de:80/outputs/rook/30398fd2-9d58-11f1-b3aa-fa163eb671ca/psl_day_EC-Earth3_dcppA-hindcast_r10i1p1f1_gr_19620901-19621130.nc
http://rook7.cloud.dkrz.de:80/outputs/rook/3039a328-9d58-11f1-b3aa-fa163eb671ca/psl_day_EC-Earth3_dcppA-hindcast_r10i1p1f1_gr_19630901-19631130.nc
http://rook7.cloud.dkrz.de:80/outputs/rook/3039b0ca-9d58-11f1-b3aa-fa163eb671ca/psl_day_EC-Earth3_dcppA-hindcast_r10i1p1f1_gr_19640901-19641130.nc


## Open the output files with xarray

The Metalink contains several batched NetCDF files. Download all of them and open them as one logical dataset with `xarray.open_mfdataset`.

In [5]:
downloaded_files = response.download()
ds = xr.open_mfdataset(
    downloaded_files,
    combine="by_coords",
    data_vars="all",
    decode_timedelta=True,
)

## Inspect the combined dataset

The xarray representation shows the dimensions, coordinates, data variables, and global attributes. The decadal metadata added by the Woodpecker fixes—including `realization`, `reftime`, and `leadtime`—is visible here.

In [6]:
ds

<xarray.Dataset> Size: 45MB
Dimensions:      (time: 273, realization: 10, lat: 57, bnds: 2, lon: 64)
Coordinates:
  * time         (time) datetime64[ns] 2kB 1962-09-01T12:00:00 ... 1964-11-30...
  * realization  (realization) int32 40B 10 1 2 3 4 5 6 7 8 9
  * lat          (lat) float64 456B 30.53 31.23 31.93 ... 68.42 69.12 69.82
  * lon          (lon) float64 512B -9.844 -9.141 -8.438 ... 33.05 33.75 34.45
    reftime      datetime64[ns] 8B 1961-11-01
    leadtime     (time) timedelta64[ns] 2kB dask.array<chunksize=(91,), meta=np.ndarray>
Dimensions without coordinates: bnds
Data variables:
    lat_bnds     (time, realization, lat, bnds) float64 2MB dask.array<chunksize=(91, 10, 57, 2), meta=np.ndarray>
    lon_bnds     (time, realization, lon, bnds) float64 3MB dask.array<chunksize=(91, 10, 64, 2), meta=np.ndarray>
    psl          (realization, time, lat, lon) float32 40MB dask.array<chunksize=(2, 23, 15, 16), meta=np.ndarray>
Attributes: (12/52)
    Conventions:                 CF-1.7 CMIP-6.2
    activity_id:                 DCPP
    branch_method:               no parent
    branch_time:                 0.0
    branch_time_in_child:        0.0
    branch_time_in_parent:       0.0
    ...                          ...
    history:                     Sat Sep 28 00:40:45 2019: ncatted -O -a vari...
    nominal_resolution:          100 km
    startdate:                   s196111
    forcing_description:         f1, CMIP6 historical forcings
    physics_description:         physics from the standard model configuratio...
    initialization_description:  Atmosphere initialization based on full-fiel...

In [7]:
ds.attrs

{'Conventions': 'CF-1.7 CMIP-6.2',
 'activity_id': 'DCPP',
 'branch_method': 'no parent',
 'branch_time': np.float64(0.0),
 'branch_time_in_child': np.float64(0.0),
 'branch_time_in_parent': np.float64(0.0),
 'contact': 'cmip6-data@ec-earth.org',
 'creation_date': '2019-05-02T10:27:28Z',
 'data_specs_version': '01.00.28',
 'experiment': 'hindcast initialized based on observations and using historical forcing',
 'experiment_id': 'dcppA-hindcast',
 'external_variables': 'areacella',
 'forcing_index': np.int32(1),
 'frequency': 'day',
 'further_info_url': 'https://furtherinfo.es-doc.org/CMIP6.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.none.r10i1p1f1',
 'grid': 'ORCA1T255',
 'grid_label': 'gr',
 'initialization_index': np.int32(1),
 'institution': 'AEMET, Spain; BSC, Spain; CNR-ISAC, Italy; DMI, Denmark; ENEA, Italy; FMI, Finland; Geomar, Germany; ICHEC, Ireland; ICTP, Italy; IDL, Portugal; IMAU, The Netherlands; IPMA, Portugal; KIT, Karlsruhe, Germany; KNMI, The Netherlands; Lund Univer